# Risk Intelligence & ML Analysis

## Context
This notebook represents the 3rd Agent in the CRIP pipeline. It is designed to evaluate multi-dimensional risk profiles, deploy machine learning for predictive pricing, and conduct stochastic simulations for stress testing portfolio solvency.

## Objectives
- Calculate comprehensive **Composite Risk Scores (1-10)** across Insurance, Credit, Market, Operational, and CAT risk dimensions.
- Train an **XGBoost Regressor** to predict expected claim amounts based on risk features and policyholder data.
- Run **Monte Carlo Simulations** to calculate the 99% Value at Risk (VaR) and assess capital adequacy.

## Concepts Used
- **Actuarial Risk Scoring**: Heuristic-based formulas combining weighted normalized inputs (e.g., loss ratios, claim frequency/severity) into 1-10 scales.
- **Predictive Modeling (XGBoost)**: Gradient boosted decision trees utilized to capture non-linear relationships between risk factors and expected financial losses.
- **Monte Carlo Simulation**: Computational algorithm that relies on repeated random sampling (e.g., normal distribution of claims) to obtain the probability of different portfolio outcomes.
- **Value at Risk (VaR)**: A statistical measure of risk that quantifies the maximum expected loss over a specific timeframe at a given confidence interval (e.g., 99%).

## Working & Methods
1. **Data Normalization**: Transforming diverse risk metrics into a standardized 0-1 range.
2. **Feature Engineering**: Deriving metrics like Claim Frequency and Claim Severity from raw policy data.
3. **Regression Training**: Fitting an XGBoost model on historical metrics to forecast future `Expected_Claim_Amount_ML`.
4. **Stochastic Sampling**: Generating thousands of portfolio loss scenarios to calculate tail risk.

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss

### 1. Load Data
We load the output from the Pricing Agent (or a raw dataset).

In [ ]:
# Assuming df is loaded from the previous step
# df = pd.read_csv('reports/pricing_output.csv')

# For demonstration, we'll create a dummy dataframe if you just run this cell
df = pd.DataFrame({'Policy_ID': [1,2], 'Sum_Insured': [100000, 200000], 'Written_Premium': [1000, 2000], 'Claim_Amount': [500, 0], 'Loss_Ratio': [0.5, 0]})
print('Data Loaded.')

In [ ]:
def normalize(series):
    '''Min-Max normalization to 0-1 range'''
    series = pd.to_numeric(series, errors='coerce').fillna(0)
    if series.max() == series.min():
        return pd.Series(0, index=series.index)
    return (series - series.min()) / (series.max() - series.min())

### 2. Actuarial Risk Formulas (1-10)
Calculate Insurance, Credit, Market, Operational, and CAT risks using defined formulas.

In [ ]:
# 1. Insurance Risk
if 'Claim_Count' not in df.columns: df['Claim_Count'] = np.random.poisson(1, len(df))
if 'Exposure_At_Risk' not in df.columns: df['Exposure_At_Risk'] = df['Sum_Insured']

df['Claim_Frequency'] = df['Claim_Count'] / df['Exposure_At_Risk'].replace(0, 1)
df['Claim_Severity'] = df['Claim_Amount'] / df['Claim_Count'].replace(0, 1)

df['Insurance_Risk'] = (
    0.4 * normalize(df['Loss_Ratio'].fillna(0)) + 
    0.3 * normalize(df['Claim_Frequency']) + 
    0.3 * normalize(df['Claim_Severity'])
) * 10

# 2. Market & Credit Risk (Dummy inputs for demo)
df['Premium_Outstanding'] = np.random.uniform(0, 5000, len(df))
df['Days_Past_Due'] = np.random.randint(0, 90, len(df))
df['Credit_Risk'] = (normalize(df['Premium_Outstanding']) * 0.6 + normalize(df['Days_Past_Due']) * 0.4) * 10

# ... Operational and CAT Risk ...
df['Fraud_Score'] = np.random.uniform(0, 100, len(df))
df['Operational_Risk'] = (0.35 * normalize(df['Fraud_Score'])) * 10

df[['Insurance_Risk', 'Credit_Risk', 'Operational_Risk']].head()

### 3. XGBoost Predictive Pricing
Train a model to predict `Expected_Claim_Amount_ML` based on features.

In [ ]:
features = ['Sum_Insured', 'Premium_Outstanding', 'Insurance_Risk', 'Credit_Risk', 'Operational_Risk']
X = df[features].fillna(0)
y_claim = df['Claim_Amount'].fillna(0)

xgb_reg = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
xgb_reg.fit(X, y_claim)
df['Expected_Claim_Amount_ML'] = xgb_reg.predict(X)

print("XGBoost trained successfully!")

### 4. Monte Carlo Simulation (VaR & Solvency)
Simulate 1000 portfolio outcomes to find the 99% Value at Risk.

In [ ]:
num_simulations = 1000
mean_loss = df['Claim_Amount'].mean()
std_loss = df['Claim_Amount'].std() if pd.notna(df['Claim_Amount'].std()) and df['Claim_Amount'].std() > 0 else mean_loss * 0.2

simulated_portfolio_losses = np.random.normal(loc=mean_loss * len(df), scale=std_loss * np.sqrt(len(df)), size=num_simulations)
var_99 = np.percentile(simulated_portfolio_losses, 99)

print(f"Portfolio VaR (99%): {var_99:,.2f}")